# get model trained on all, ensemble trained on all, ensemble refined

### imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Load config
import sys
import os
from pathlib import Path
from IPython.display import display


# Add the parent directory to the path so we can import modules properly
cwd = Path.cwd()
print(f"home directory: {cwd}")
relative_repo_path = "GitRepos/simulation_closed_loop"

# append repo path 
sys.path.append(str(cwd / relative_repo_path))

from model_in_the_loop.utils.hydra_utils import load_config,set_env_vars
cfg = load_config()
set_env_vars(cfg)  # set env variables for repo and data paths


home directory: /gpfs01/euler/User/ssuhai


In [3]:
cfg.model_configs.paths

{'load_model_path': '/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/models/full_ensemble', 'cache_dir': '${oc.env:OPENRETINA_CACHE_DIRECTORY}', 'data_dir': '${paths.cache_dir}/data/', 'log_dir': '.', 'output_dir': '${hydra:runtime.output_dir}', 'movies_path': 'https://huggingface.co/datasets/open-retina/open-retina/blob/main/euler_lab/hoefling_2024/stimuli/rgc_natstim_18x16_joint_normalized_2024-01-11.zip', 'responses_path': 'https://huggingface.co/datasets/open-retina/open-retina/resolve/main/euler_lab/hoefling_2024/responses/rgc_natstim_2024-08-14.zip'}

In [4]:
from model_in_the_loop.core.dj_wrappers import (DJTableHolder,Preprocessor,QualityAndTypeWrapper,STAWrapper,RandomSeedMEIWrapper)

from model_in_the_loop.utils.file_management import copy_rec_files,create_directory_structure,rm_all_experiment_dirs,clear_data_dump_dir
from model_in_the_loop.utils.transform_to_avi_stimulus import create_single_mei_avis_and_metadata,create_rf_test_dir
from model_in_the_loop.utils.simple_logging import log
from model_in_the_loop.utils.plotting import show_all_rois_plot

### Create processing components (connect them to DB)

In [5]:
# create preprocessor

dj_table_holder = DJTableHolder(
                username=cfg.DJ.username, # type: ignore
                
                #paths
                home_directory=cfg.paths.home_directory, # type: ignore
                repo_directory=cfg.paths.repo_directory, # type: ignore
                dj_config_directory= cfg.paths.dj_config_directory, # type: ignore
                rgc_output_directory= cfg.paths.rgc_output_directory, # type: ignore

                userinfo= cfg.DJ.userinfo, # type: ignore

                table_parameters=cfg.DJ.table_parameters, # type: ignore

                # from overall configs
                debug=cfg.debug, # type: ignore
                plot_results=cfg.plot_results, # type: ignore
                )



In [6]:
dj_table_holder.setup()


[2025-10-06 13:45:14,167][INFO]: Connecting ssuhai@172.25.240.200:3306
[2025-10-06 13:45:14,219][INFO]: Connected ssuhai@172.25.240.200:3306


schema_name: ageuler_ssuhai_closed_loop
Done reconnecting. Skipping adding new entries from config.


/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/model_in_the_loop/core/dj_wrappers/base.py:293: UserWarning: 
Some DJ tables (like UserInfo) are not empty, skipping adding new entries from config.
Make sure this is wanted. Call clear_tables(`all`) if you want different data in there
  warnings.warn("\nSome DJ tables (like UserInfo) are not empty, skipping adding new entries from config.\nMake sure this is wanted. Call clear_tables(`all`) if you want different data in there")


In [25]:
preprocessor = Preprocessor(dj_table_holder=dj_table_holder)


quality_type_analysis_wrapper = QualityAndTypeWrapper(
    dj_table_holder=dj_table_holder,)

sta_wrapper = STAWrapper(
    dj_table_holder=dj_table_holder,)

## get data

### Move files from server to the repo 

In [26]:
# create_directory_structure(base_directory= cfg.DJ.userinfo.data_dir,)

# copy_rec_files(
#     recording_files_dir=cfg.paths.recording_files_dir,  # type: ignore
#     destination_base=cfg.DJ.userinfo.data_dir,  # type: ignore
#     full_dummy_ini_dir= os.path.join(cfg.paths.repo_directory, cfg.paths.dummy_ini_dir),  # type: ignore
# )

### Preprocessing

In [13]:
preprocessor.upload_iteration_metadata()

In [27]:
missing_keys = dj_table_holder("RoiMask")().list_missing_field()
if len(missing_keys) == 1:
    field_key = missing_keys[0]
    print(f"Missing field key found: {field_key}")
elif len(missing_keys) > 1:
    raise ValueError(f"Multiple missing fields found: {missing_keys}")
else:
    print("No missing fields found, using the last field key.")
    all_field_key = dj_table_holder("Field")().proj().fetch(as_dict=True)
    field_key = all_field_key[-1]
    print(f"Field key: {field_key}")

In [15]:
# compute 
preprocessor.add_iteration_roi_mask(field_key=field_key)
preprocessor.add_iteration_rois()
preprocessor.add_iteration_traces()

### qualty and RF

In [28]:
quality_type_analysis_wrapper.compute_analysis(
    field_key=field_key)

# filter 
passed_roi_ids_chirp_mb = quality_type_analysis_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
    d_qi_min =cfg.quality_filtering["d_qi_min"],
    qidx_min=cfg.quality_filtering["chirp_qi_min"],
    celltypes=cfg.quality_filtering["celltypes"],
    classifier_confidence=cfg.quality_filtering["classifier_confidence"])
if len(passed_roi_ids_chirp_mb) == 0:
    raise ValueError("No ROIs passed the quality criterion for quality and type.")
print(f"{len(passed_roi_ids_chirp_mb)} ROIs passed quality chirp mb filtering: {passed_roi_ids_chirp_mb}")



In [29]:
# sta 
sta_wrapper.compute_analysis(
    field_key=field_key,
    roi_id_subset=passed_roi_ids_chirp_mb,)


In [30]:
assert ((dj_table_holder("STA")() & field_key).fetch("roi_id") == passed_roi_ids_chirp_mb).all(), "STA roi_id does not match passed roi_ids from quality and type filtering."

In [99]:
# filter
passed_roi_ids_sta = sta_wrapper.get_roi_ids_passing_criterion(field_key=field_key,
                                                               rf_qidx_min= cfg.quality_filtering["rf_qidx_min"])
if len(passed_roi_ids_sta) == 0:
    raise ValueError("No ROIs passed the quality criterion for STA.")
print(f"{len(passed_roi_ids_sta)} ROIs passed STA filtering with rf_qidx_min={cfg.quality_filtering["rf_qidx_min"]}: {passed_roi_ids_sta}")

if len(passed_roi_ids_sta) < 25:
    print("HAVE AT LEAST 25 ROIS")


In [ ]:
# if len(passed_roi_ids_sta) >= 25:
#     print("MORE THAN 25 ROIS, selecting 25 best")
#     top_n_rois_sta,top_n_scores = sta_wrapper.list_top_n_rois_by_qidx(field_key=field_key,
#                                                     n=25,)
#     passed_roi_ids_sta = top_n_rois_sta
#     print(top_n_rois_sta)
#     print(top_n_scores)
#     print(f"Using top 25 rois based on rf_qidx: {passed_roi_ids_sta}")

# get wrappers

In [110]:
def get_desired_wrapper(cfg,mode,field_key,roi_id_subset):
    if mode == "single_member_train_full":
        cfg.model_configs.only_train_readout = False
        cfg.model_configs.is_ensemble_model = False
    elif mode == "refine_ensemble":
        cfg.model_configs.only_train_readout = True
        cfg.model_configs.is_ensemble_model = True
        print(cfg.model_configs.paths.load_model_path)
    else:
        raise ValueError(f"Mode {mode} not recognized.")



    random_seed_mei_wrapper = RandomSeedMEIWrapper(
    dj_table_holder=dj_table_holder,
    cfg=cfg,
    seeds=[111,222]
    )

    # compute 
    random_seed_mei_wrapper.compute_analysis(
        field_key=field_key,
        roi_id_subset=roi_id_subset,
        )

    return random_seed_mei_wrapper


In [117]:
mei_wrapper_all = get_desired_wrapper(cfg=cfg,mode="single_member_train_full",field_key=field_key,roi_id_subset=passed_roi_ids_sta)

In [120]:
mei_wrapper_all.save_all_data_to_dir(mei_wrapper_all.save_dir_parent)

In [132]:
1 + 1
mei_wrapper_refine_ensemble = get_desired_wrapper(cfg=cfg,mode="refine_ensemble",field_key=field_key,roi_id_subset=passed_roi_ids_sta)

## visualize with GUI

In [ ]:
from model_in_the_loop.core.gui import ExtendedAutoRoiGui
gui = ExtendedAutoRoiGui(
    dj_table_holder=dj_table_holder, 
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper,mei_wrapper_all],
    #take_roi_rgba_from_this_analysis=quality_type_analysis_wrapper.name,
    take_roi_rgba_from_this_analysis=mei_wrapper_all.name,
    # JUST VIS
    do_not_compute_only_visualize = True,
    
    field_key=field_key,
    canvas_width=30)

In [119]:
display(gui.start_gui())

In [ ]:
gui = ExtendedAutoRoiGui(
    dj_table_holder=dj_table_holder, 
    dj_preprocessor=preprocessor,
    all_dj_wrappers=[quality_type_analysis_wrapper,sta_wrapper,mei_wrapper_refine_ensemble],
    #take_roi_rgba_from_this_analysis=quality_type_analysis_wrapper.name,
    take_roi_rgba_from_this_analysis=mei_wrapper_refine_ensemble.name,
    # JUST VIS
    do_not_compute_only_visualize = True,
    
    field_key=field_key,
    canvas_width=30)



In [135]:
display(gui.start_gui())

In [136]:
mei_wrapper_refine_ensemble.save_all_data_to_dir(mei_wrapper_refine_ensemble.save_dir_parent)

In [138]:
def get_mei_order(wrapper, wrapper_name):
    roi_id2mei_ids, roi_id2info = wrapper.select_subset_of_meis_for_each_roi(
                                            neuron_data_dict =wrapper.neuron_data_dict,
                                            new_session_id =wrapper.new_session_id,
                                            mei_data_container = wrapper.mei_data_container,
                                            readout_idx_wmei2rois = wrapper.readout_idx_wmei2rois,
                                            neuron_idxs_passing_filter = wrapper.neuron_idxs_passing_filter,
                                            n_stimuli_total = 6,
    )
    parent_dir ="/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis"
    # save 
    import pickle 
    with open(os.path.join(parent_dir,f"{wrapper_name}_roi_id2mei_ids.pkl"),"wb") as f:
        pickle.dump(roi_id2mei_ids,f)
    with open(os.path.join(parent_dir,f"{wrapper_name}_roi_id2info.pkl"),"wb") as f:
        pickle.dump(roi_id2info,f)
    return roi_id2mei_ids, roi_id2info



# ensemble_roi_id2mei_ids, ensemble_roi_id2info = get_mei_order(mei_wrapper_refine_ensemble,"refine_ensemble")
# member_roi_id2mei_ids, member_roi_id2info = get_mei_order(mei_wrapper_all,"single_member_train_full")

In [149]:
def save_strfs():
    roi_id,srf,trf = (dj_table_holder("SplitRF")() & "cond1='iter1'").fetch("roi_id","srf","trf")
    parent_dir ="/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis"
    import pickle 
    with open(os.path.join(parent_dir,"roi_id_srf_trf.pkl"),"wb") as f:
        pickle.dump((roi_id,srf,trf),f)

save_strfs()

In [151]:
# dj_table_holder("CelltypeAssignment")()

In [ ]:
def save_celltype():
    parent_dir ="/gpfs01/euler/User/ssuhai/GitRepos/simulation_closed_loop/thesis/data/validate_online_meis"
    
    

In [ ]:
def get_data_container_fully_trained_ensemble(model):
    import torch
    from model_in_the_loop.utils.stimulus_optimization import generate_opt_stim_for_neuron_list
    from model_in_the_loop.utils.stimulus_optimization import decompose_mei,reconstruct_mei_from_decomposed
    from model_in_the_loop.utils.model_training import BaseCoreRead
    # initialize mei data containter
    mei_data_container_entries = []

    ## generate meis
    for phase in ['unstable', 'stable']:
        print(f"Generating {phase} MEIs for neurons (readout idx): {[idx for idx,stab in idx2stability.items() if stab ==phase ]}.")
        log(f"Generating {phase} MEIs for neurons (readout idx): {[idx for idx,stab in idx2stability.items() if stab == phase ]}.")
        set_model_to_eval_mode = True if phase == 'stable' else False
        neuron_ids_to_analyze = [neuron_id for neuron_id, stability in idx2stability.items() if stability == phase]
        seeds = self.seeds if phase == 'unstable' else [self.seeds[0]] # only one seed for stable meis

        neuron_seed_mei_dict =  generate_opt_stim_for_neuron_list(
                                        model = self.model,
                                        new_session_id = self.new_session_id,
                                        opt_stim_generation_params= self.mei_generation_params,
                                        random_seeds = seeds,
                                        seed_it_func= torch.manual_seed,
                                        neuron_ids_to_analyze = neuron_ids_to_analyze, # NOTE: this will optimize each id individually 
                                        set_model_to_eval_mode = set_model_to_eval_mode, # model in training mode for noisy MEIs
                                        )
        print(f"Done with meis in phase {phase}.")
        print(f"Start decomposing ...")    
        
        ## decompose meis
        device = "cuda"
        for neuron_id,seed_dict in neuron_seed_mei_dict.items():
            print(f"Decomposing MEIs for neuron (readout idx) {neuron_id} ...")
            for seed,mei in seed_dict.items():

                # decompose the MEIs
                temporal_kernels, spatial_kernels, _ = decompose_mei(stimulus = mei.detach().cpu().numpy())
            

                if self.mei_generation_params["reconstruct_mei"]:
                    reconstruction = reconstruct_mei_from_decomposed(
                                temporal_kernels=temporal_kernels,
                                spatial_kernels=spatial_kernels,
                                turn_to_tensor=True)

                    assert reconstruction.shape == mei.shape, "Reconstructed MEI shape does not match original MEI shape."
                    
                    # make reonstruction same norm as mei
                    print(f"changing norm of reconstruction {torch.norm(reconstruction)} to match original mei norm {torch.norm(mei)}")
                    reconstruction = reconstruction / torch.norm(reconstruction) * torch.norm(mei)
                    print(f"new reconstruction norm {torch.norm(reconstruction)}")
                    mei = reconstruction # use the reconstructed MEI for further analysis
                    print(f"Done reconstructing MEI for neuron (readout idx) {neuron_id}, seed {seed}.")
                    # add entry to data container 
                    mei_data_container_entries.append({
                        "readout_idx": neuron_id,
                        "roi_id": self.readout_idx_wmei2rois[neuron_id],
                        "mei_id": f"roi_{self.readout_idx_wmei2rois[neuron_id]}_seed_{seed}",
                        "seed": seed,
                        "mei": mei.detach(),
                        "temporal_kernels": temporal_kernels,
                        "spatial_kernels": spatial_kernels,
                        "stability": phase,
                    })
        

                    

    # make df container from all meis
    self.mei_data_container = pd.DataFrame(mei_data_container_entries)

    print(f"Generating responses for neurons in readout {len(self.neuron_idxs_passing_filter)} to all meis {len(self.mei_data_container)} ...")
    self.get_responses_and_add_to_container(mei_data_container=self.mei_data_container,
                                            model=self.model,
                                            new_session_id=self.new_session_id,
                                            neuron_data_dict=self.neuron_data_dict,
                                            

# Plot 

In [ ]:
def plot_optimized_kernel_and_trf()